# Лабораторна робота 5. Власне донавчання: класифікація українських відгуків

## 1.1. Задача

**Прикладна задача:** автоматична класифікація українських відгуків та звернень користувачів за двома параметрами одночасно — **емоційним забарвленням (`final_emotion`)** та **категорією звернення (`final_category`)** — із поверненням результату у вигляді чистого JSON без зайвого тексту.

**Як використовуватиметься на практиці:**
- **Користувач системи:** відділ підтримки клієнтів, продуктова аналітика або система автоматичної маршрутизації звернень українського сервісу/маркетплейсу.
- **Вхід:** довільний текстовий відгук або коментар користувача українською мовою.
- **Вихід:** валідний JSON-обʼєкт формату `{"final_emotion": "<емоція>", "final_category": "<категорія>"}`, який парситься бекендом для пріоритезації негативних відгуків і направлення у потрібний відділ.

**Чому потрібне саме донавчання, а не промпт чи RAG:**
1. **Вартість і затримка:** без донавчання в кожен запит доводиться передавати повний перелік дозволених міток та інструкції форматування (168 токенів системного промпту замість 37), що збільшує витрату токенів на тисячах відгуків щодня.
2. **Стабільність формату:** загальні instruct-моделі часто огортають відповідь у Markdown (` ```json ... ``` `) або додають пояснення.
3. **Розрізнення близьких категорій:** SFT закріплює межі між суміжними категоріями та емоціями безпосередньо у вагах адаптера. RAG тут неефективний, оскільки задача полягає не в пошуку фактів у базі знань, а у дотриманні фіксованої схеми класифікації.

## 1.2. Модель і дані

- **Модель:** `Qwen/Qwen3-1.7B` (Instruct, 1.7 млрд параметрів). Обрана як оптимальний баланс якості розуміння української мови, швидкості інференсу та споживання відеопамʼяті (вміщується у 15 GB GPU T4 із LoRA у `fp16` без квантизації). Використовується Instruct-версія, оскільки вона вже володіє шаблоном розмови ChatML і потребує лише адаптації під вузьку схему класифікації.
- **Дані:** датасет [`KSE-RESEARCH-Group/UAReviews`](https://huggingface.co/datasets/KSE-RESEARCH-Group/UAReviews) (Hugging Face Hub, ліцензія **CC BY 4.0**, 11 580 реальних українських відгуків та коментарів користувачів, розмічених за емоцією та наміром).

### Встановлення бібліотек та перевірка GPU

In [1]:
!pip install -q "transformers>=4.55.0" "trl>=1.17" "peft" "datasets" "accelerate" "scikit-learn"
!pip uninstall -q -y torchao

import gc
import json
import os
import time
import pandas as pd
import torch
from datasets import load_dataset
from peft import LoraConfig, PeftModel
from sklearn.metrics import f1_score
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer

assert torch.cuda.is_available(), "Потрібна GPU! Увімкніть T4 у Runtime -> Change runtime type."
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Доступна памʼять GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU: Tesla T4
Доступна памʼять GPU: 15.6 GB


### Завантаження, очищення, дедуплікація та поділ даних

In [3]:
import json

raw_ds = load_dataset("KSE-RESEARCH-Group/UAReviews", split="train")
print("Колонки датасету:", raw_ds.column_names)
print("Кількість сирих прикладів:", len(raw_ds))

# Явно задаємо колонки з датасету
text_col = "content"
label_cols = ["final_emotion", "final_category"]
print(f"Текстова колонка: '{text_col}', колонки міток: {label_cols}")

# Фільтруємо порожні значення та дедуплікуємо за текстом
seen, keep = set(), []
for i, row in enumerate(raw_ds):
    txt = str(row[text_col]).strip()
    l1, l2 = row[label_cols[0]], row[label_cols[1]]
    if not txt or l1 is None or l2 is None:
        continue
    key = txt.lower()
    if key not in seen:
        seen.add(key)
        keep.append(i)

dedup_ds = raw_ds.select(keep)
print(f"Після очищення та дедуплікації: {len(dedup_ds)}")

# Отримуємо текстові назви класів (якщо колонки у форматі ClassLabel)
def get_label_str(ds, col, val):
    feat = ds.features[col]
    return feat.int2str(val) if hasattr(feat, "int2str") else str(val)

labels_1 = sorted({get_label_str(dedup_ds, label_cols[0], v) for v in dedup_ds[label_cols[0]]})
labels_2 = sorted({get_label_str(dedup_ds, label_cols[1], v) for v in dedup_ds[label_cols[1]]})
print(f"{label_cols[0]} ({len(labels_1)}): {labels_1}")
print(f"{label_cols[1]} ({len(labels_2)}): {labels_2}")

SHORT_PROMPT = "Класифікуй український відгук користувача. Відповідай лише валідним JSON."

def to_messages(example):
    target_obj = {
        label_cols[0]: get_label_str(dedup_ds, label_cols[0], example[label_cols[0]]),
        label_cols[1]: get_label_str(dedup_ds, label_cols[1], example[label_cols[1]]),
    }
    return {
        "messages": [
            {"role": "system", "content": SHORT_PROMPT},
            {"role": "user", "content": str(example[text_col]).strip()},
            {"role": "assistant", "content": json.dumps(target_obj, ensure_ascii=False)},
        ],
        "chat_template_kwargs": {"enable_thinking": False},
    }

shuffled = dedup_ds.shuffle(seed=42)
# Беремо 2500 на train+val (2250 train / 250 val) та 500 на незалежний test
train_val = shuffled.select(range(2500)).map(to_messages, remove_columns=shuffled.column_names)
split_tv = train_val.train_test_split(test_size=250, seed=42)
train_dataset, val_dataset = split_tv["train"], split_tv["test"]
test_dataset = shuffled.select(range(2500, 3000)).map(to_messages, remove_columns=shuffled.column_names)

print(f"\nПоділ -> Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")
print("\nПриклад форматованого діалогу:")
for msg in train_dataset[0]["messages"]:
    print(f"  [{msg['role']}]: {msg['content']}")

Колонки датасету: ['id', 'rating', 'content', 'source', 'final_emotion', 'final_category', 'length', 'split']
Кількість сирих прикладів: 11580
Текстова колонка: 'content', колонки міток: ['final_emotion', 'final_category']
Після очищення та дедуплікації: 11564
final_emotion (7): ['Anger', 'Disgust', 'Fear', 'Happiness', 'Neutral', 'Sadness', 'Surprise']
final_category (5): ['Complaint / Dissatisfaction', 'Gratitude / Positive Feedback', 'Neutral Comment', 'Question / Request for Help', 'Suggestion / Idea']


Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]


Поділ -> Train: 2250, Val: 250, Test: 500

Приклад форматованого діалогу:
  [system]: Класифікуй український відгук користувача. Відповідай лише валідним JSON.
  [user]: А від кримінальної справи інтерполу також звільняєте?
  [assistant]: {"final_emotion": "Neutral", "final_category": "Question / Request for Help"}


## 1.3. Модель до донавчання

**Обрані метрики оцінювання:**
1. **Чистий JSON (%)** — частка відповідей, які одразу парсяться як JSON без обгорток на кшталт ` ```json ` чи вступних слів. Критично для автоматичної інтеграції в бекенд.
2. **Валідний JSON (%)** — частка відповідей, які після зняття Markdown-обгортки містять обидва очікувані ключі.
3. **Точність за кожним полем (Accuracy, %)** — частка точних збігів за першою та другою міткою на всьому тестовому наборі (невалідний JSON рахується як помилка).
4. **Macro F1 (%)** — середнє F1 по всіх класах кожної мітки; враховує можливий дисбаланс класів у відгуках.

До донавчання передаємо моделі максимально детальний промпт (`LONG_PROMPT`), у якому перелічено всі допустимі значення міток і формат виводу.

### Функції оцінювання та базова оцінка моделі до донавчання

In [4]:
MODEL_NAME = "Qwen/Qwen3-1.7B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

LONG_PROMPT = f"""Класифікуй український відгук користувача для системи аналітики та маршрутизації.
Відповідай ОДНИМ обʼєктом JSON без жодного іншого тексту чи розмітки Markdown, у форматі:
{{"{label_cols[0]}": "<значення>", "{label_cols[1]}": "<значення>"}}

Дозволені значення для "{label_cols[0]}": {", ".join(labels_1)}
Дозволені значення для "{label_cols[1]}": {", ".join(labels_2)}"""

print(f"Довжина SHORT_PROMPT: {len(tokenizer(SHORT_PROMPT)['input_ids'])} токенів")
print(f"Довжина LONG_PROMPT:  {len(tokenizer(LONG_PROMPT)['input_ids'])} токенів")

def classify_batch(model, texts_batch, system_prompt, max_new_tokens=64):
    prompts = [
        tokenizer.apply_chat_template(
            [{"role": "system", "content": system_prompt}, {"role": "user", "content": text}],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
        for text in texts_batch
    ]
    tokenizer.padding_side = "left"
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    tokenizer.padding_side = "right"
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    new_tokens = outputs[:, inputs["input_ids"].shape[1]:]
    return [ans.strip() for ans in tokenizer.batch_decode(new_tokens, skip_special_tokens=True)]

def is_clean_json(answer):
    try:
        res = json.loads(answer)
        return isinstance(res, dict) and set(label_cols) <= res.keys()
    except json.JSONDecodeError:
        return False

def parse_json(answer):
    text = answer.strip()
    if text.startswith("```"):
        text = text.strip("`").removeprefix("json").strip()
    try:
        res = json.loads(text)
    except json.JSONDecodeError:
        return None
    if isinstance(res, dict) and set(label_cols) <= res.keys():
        return res
    return None

def evaluate_model(model, system_prompt, batch_size=32):
    model.eval()
    texts = [ex["messages"][1]["content"] for ex in test_dataset]
    targets = [json.loads(ex["messages"][2]["content"]) for ex in test_dataset]
    answers = []
    for i in range(0, len(texts), batch_size):
        answers.extend(classify_batch(model, texts[i:i + batch_size], system_prompt))

    clean = valid = ok_1 = ok_2 = 0
    y_true_1, y_pred_1 = [], []
    y_true_2, y_pred_2 = [], []

    for ans, tgt in zip(answers, targets):
        clean += int(is_clean_json(ans))
        parsed = parse_json(ans)
        t1, t2 = str(tgt[label_cols[0]]), str(tgt[label_cols[1]])
        y_true_1.append(t1)
        y_true_2.append(t2)
        if parsed is not None:
            valid += 1
            p1, p2 = str(parsed[label_cols[0]]), str(parsed[label_cols[1]])
            ok_1 += int(p1 == t1)
            ok_2 += int(p2 == t2)
            y_pred_1.append(p1)
            y_pred_2.append(p2)
        else:
            y_pred_1.append("__INVALID__")
            y_pred_2.append("__INVALID__")

    n = len(answers)
    metrics = {
        "Чистий JSON, %": round(100 * clean / n, 2),
        "Валідний JSON, %": round(100 * valid / n, 2),
        f"Accuracy ({label_cols[0]}), %": round(100 * ok_1 / n, 2),
        f"Accuracy ({label_cols[1]}), %": round(100 * ok_2 / n, 2),
        f"Macro F1 ({label_cols[0]}), %": round(100 * f1_score(y_true_1, y_pred_1, average="macro", zero_division=0), 2),
        f"Macro F1 ({label_cols[1]}), %": round(100 * f1_score(y_true_2, y_pred_2, average="macro", zero_division=0), 2),
    }
    return metrics, list(zip(texts, targets, answers))

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float16, device_map="auto")
print(f"Памʼять GPU після завантаження моделі: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

before_metrics, before_samples = evaluate_model(base_model, LONG_PROMPT)
print("\nМетрики ДО донавчання (з детальним промптом):")
display(pd.Series(before_metrics))

print("\nПриклади відповідей ДО донавчання (перші 7 із тесту):")
for txt, tgt, ans in before_samples[:7]:
    print(f"Відгук: {txt}\nЕталон: {json.dumps(tgt, ensure_ascii=False)}\nМодель: {ans}\n" + "-" * 70)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (5 outputs left out)

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Памʼять GPU після завантаження моделі: 3.44 GB

Метрики ДО донавчання (з детальним промптом):


,0
"Чистий JSON, %",100.00
"Валідний JSON, %",100.00
"Accuracy (final_emotion), %",76.80
"Accuracy (final_category), %",90.20
"Macro F1 (final_emotion), %",24.88
"Macro F1 (final_category), %",59.00



Приклади відповідей ДО донавчання (перші 7 із тесту):
Відгук: Дуже впорядкували все, чисто, тепло, поставили пандуси.
Еталон: {"final_emotion": "Happiness", "final_category": "Gratitude / Positive Feedback"}
Модель: {"final_emotion": "Happiness", "final_category": "Gratitude / Positive Feedback"}
----------------------------------------------------------------------
Відгук: Якщо маєте можливість, зверніться у будь-який інший відділ. Керівниця не тільки не компетентна, а й дозволяє собі кричати на відвідувачів, у тому числі на матір військовослужбовця, що зник безвісті
Еталон: {"final_emotion": "Anger", "final_category": "Complaint / Dissatisfaction"}
Модель: {"final_emotion": "Anger", "final_category": "Complaint / Dissatisfaction"}
----------------------------------------------------------------------
Відгук: Напередодні зареєструвалась в онлайн черзі. Мене прийняли як і зазначалось в квитку о 12-00. За 20 хвилин оформили документи на отримання грошей на пакунок малюка, тому я дуже з

## 1.4. Донавчання (SFT + LoRA)

**Обґрунтування вибору методу та гіперпараметрів:**
- **Метод:** SFT із **LoRA** (`r=16`, `lora_alpha=32`, `lora_dropout=0.05`, `target_modules="all-linear"`). Повне донавчання 1.7B моделі у `fp32` з оптимізатором Adam потребує >25 GB VRAM, що перевищує ліміт T4 (15 GB). LoRA тренує ~1% параметрів у всіх лінійних шарах (attention + MLP), зберігаючи загальні знання мови.
- **`learning_rate=2e-4`:** стандартне значення для LoRA (приблизно у 10 разів вище, ніж для повного донавчання).
- **Ефективний розмір батча = 16** (`per_device_train_batch_size=4`, `gradient_accumulation_steps=4`): забезпечує стабільні градієнти при мінімальному споживанні памʼяті.
- **`max_steps=150`** (~1.07 епохи на 2250 прикладах) із косинусним планувальником (`cosine`) та `warmup_steps=10`.
- **`assistant_only_loss=True`:** функція втрат обчислюється виключно на токенах цільового JSON-відгуку асистента, щоб модель не витрачала ємність адаптера на запамʼятовування вхідних відгуків.
- **`fp16=True` та `gradient_checkpointing=True`:** оптимально для архітектури NVIDIA T4.

### Тренування LoRA та лог Training/Validation Loss

In [5]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

sft_config = SFTConfig(
    output_dir="qwen3-1.7b-ua-reviews-lora",
    max_length=256,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    max_steps=150,
    eval_strategy="steps",
    eval_steps=15,
    logging_steps=15,
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    assistant_only_loss=True,
    save_strategy="no",
    report_to="none",
)

torch.cuda.reset_peak_memory_stats()
trainer = SFTTrainer(
    model=base_model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

trainer.model.print_trainable_parameters()

start_time = time.time()
trainer.train()
train_minutes = (time.time() - start_time) / 60
peak_vram_gb = torch.cuda.max_memory_allocated() / 1e9

print(f"\nТривалість тренування: {train_minutes:.2f} хв")
print(f"Пікова памʼять GPU: {peak_vram_gb:.2f} GB")

# Зберігаємо адаптер локально (не додаємо його у Git-репозиторій!)
ADAPTER_DIR = "qwen3-1.7b-ua-reviews-adapter"
trainer.save_model(ADAPTER_DIR)
adapter_size_mb = os.path.getsize(os.path.join(ADAPTER_DIR, "adapter_model.safetensors")) / 1e6
print(f"Розмір збереженого адаптера: {adapter_size_mb:.1f} MB")

# Таблиця динаміки Training та Validation Loss
logs = trainer.state.log_history
train_logs = {h["step"]: h["loss"] for h in logs if "loss" in h}
eval_logs = {h["step"]: h["eval_loss"] for h in logs if "eval_loss" in h}
loss_df = pd.DataFrame([
    {"Step": s, "Training Loss": train_logs.get(s), "Validation Loss": eval_logs.get(s)}
    for s in sorted(set(train_logs) | set(eval_logs))
])
display(loss_df.round(4))

Tokenizing train dataset:   0%|          | 0/2250 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/2250 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2250 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/2250 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/250 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/250 [00:00<?, ? examples/s]

... (2 outputs left out)

trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
15,2.644947,0.147021,0.124666,0.969241,35423.000000
30,0.088292,0.041631,0.042660,0.988689,70567.000000
45,0.037789,0.034360,0.044974,0.990474,105665.000000
60,0.029704,0.028779,0.027807,0.991341,142032.000000
75,0.024915,0.027319,0.024681,0.991162,177436.000000
90,0.027338,0.024862,0.032610,0.991519,213942.000000
105,0.019292,0.023409,0.025986,0.992068,250017.000000
120,0.024391,0.023412,0.025474,0.991873,285026.000000
135,0.023415,0.023694,0.027935,0.992432,321347.000000
150,0.022054,0.023575,0.027518,0.992611,357865.000000



Тривалість тренування: 9.63 хв
Пікова памʼять GPU: 4.08 GB
Розмір збереженого адаптера: 69.8 MB


,Step,Training Loss,Validation Loss
0,15,2.6449,0.1470
1,30,0.0883,0.0416
2,45,0.0378,0.0344
3,60,0.0297,0.0288
4,75,0.0249,0.0273
5,90,0.0273,0.0249
6,105,0.0193,0.0234
7,120,0.0244,0.0234
8,135,0.0234,0.0237
9,150,0.0221,0.0236


## 1.5. Порівняння до і після донавчання та аналіз помилок

Оцінюємо донавчену модель на **тому самому тестовому наборі (500 прикладів)** і тих самих 7 прикладах, але тепер передаємо лише **короткий системний промпт (`SHORT_PROMPT`)** без переліку категорій. Також перевіряємо, чи зберегла модель здатність відповідати на загальні запити українською мовою поза цільовою задачею.

### Оцінка після донавчання, порівняльна таблиця, аналіз помилок і тест загальних знань

In [6]:
after_metrics, after_samples = evaluate_model(trainer.model, SHORT_PROMPT)

comparison_df = pd.DataFrame({
    "До донавчання (LONG_PROMPT)": before_metrics,
    "Після LoRA (SHORT_PROMPT)": after_metrics,
})
comparison_df["Різниця"] = (comparison_df["Після LoRA (SHORT_PROMPT)"] - comparison_df["До донавчання (LONG_PROMPT)"]).round(2)
display(comparison_df)

print("\nПорівняння на тих самих 7 тестових прикладах:")
for (txt, tgt, ans_before), (_, _, ans_after) in zip(before_samples[:7], after_samples[:7]):
    print(f"Відгук: {txt}")
    print(f"  Еталон: {json.dumps(tgt, ensure_ascii=False)}")
    print(f"  До:     {ans_before}")
    print(f"  Після:  {ans_after}")
    print("-" * 70)

# Аналіз помилок донавченої моделі
errors = []
for txt, tgt, ans in after_samples:
    parsed = parse_json(ans)
    if parsed is None or str(parsed[label_cols[0]]) != str(tgt[label_cols[0]]) or str(parsed[label_cols[1]]) != str(tgt[label_cols[1]]):
        errors.append((txt, tgt, ans))

print(f"\nКількість помилок на 500 тестових прикладах: {len(errors)} ({100 * len(errors) / len(after_samples):.1f}%)")
print("Приклади помилок (до 8 штук):")
for txt, tgt, ans in errors[:8]:
    print(f"Відгук: {txt}\n  Очікувалося: {json.dumps(tgt, ensure_ascii=False)}\n  Отримано:    {ans}\n" + "-" * 60)

# Перевірка на загальних запитах поза задачею (чи немає катастрофічного забування)
general_prompts = [
    "Поясни простими словами, що таке overfitting у машинному навчанні.",
    "Напиши три поради для ефективного планування робочого дня.",
    "Яка столиця Франції та чим вона відома?",
]
print("\nПеревірка збереження загальних навичок моделі:")
for q in general_prompts:
    ans = classify_batch(trainer.model, [q], "Ти — корисний україномовний асистент.", max_new_tokens=120)[0]
    print(f"Питання: {q}\nВідповідь: {ans}\n" + "-" * 60)

,До донавчання (LONG_PROMPT),Після LoRA (SHORT_PROMPT),Різниця
"Чистий JSON, %",100.00,100.00,0.00
"Валідний JSON, %",100.00,100.00,0.00
"Accuracy (final_emotion), %",76.80,91.40,14.60
"Accuracy (final_category), %",90.20,92.20,2.00
"Macro F1 (final_emotion), %",24.88,42.06,17.18
"Macro F1 (final_category), %",59.00,75.08,16.08



Порівняння на тих самих 7 тестових прикладах:
Відгук: Дуже впорядкували все, чисто, тепло, поставили пандуси.
  Еталон: {"final_emotion": "Happiness", "final_category": "Gratitude / Positive Feedback"}
  До:     {"final_emotion": "Happiness", "final_category": "Gratitude / Positive Feedback"}
  Після:  {"final_emotion": "Happiness", "final_category": "Gratitude / Positive Feedback"}
----------------------------------------------------------------------
Відгук: Якщо маєте можливість, зверніться у будь-який інший відділ. Керівниця не тільки не компетентна, а й дозволяє собі кричати на відвідувачів, у тому числі на матір військовослужбовця, що зник безвісті
  Еталон: {"final_emotion": "Anger", "final_category": "Complaint / Dissatisfaction"}
  До:     {"final_emotion": "Anger", "final_category": "Complaint / Dissatisfaction"}
  Після:  {"final_emotion": "Anger", "final_category": "Complaint / Dissatisfaction"}
----------------------------------------------------------------------
Відгук:

## 1.6. Мій звіт

- **Задача і чому потрібне донавчання:** Автоматична класифікація українських відгуків користувачів за двома ознаками (`final_emotion` та `final_category`) із виводом суворого JSON-обʼєкта для систем підтримки та аналітики. Донавчання потрібне, щоб прибрати довгий системний промпт із переліком усіх міток у кожному запиті, гарантувати 100% чистий JSON без зайвого тексту і навчити модель краще розрізняти межі класів українською мовою.
- **Модель і дані:** Модель `Qwen/Qwen3-1.7B` (Instruct). Датасет `KSE-RESEARCH-Group/UAReviews` (ліцензія CC BY 4.0, 11 580 українських відгуків). Після очищення та дедуплікації (11 564 записи) сформовано вибірки у форматі `messages`: тренувальна — 2 250 прикладів, валідаційна — 250 прикладів, тестова — 500 прикладів (без перетину з тренуванням).
- **Метрика й результат до донавчання:** Використано частку чистого JSON, частку валідного JSON, Accuracy та Macro F1 по обох полях. До донавчання навіть із детальним промптом (`LONG_PROMPT`, 168 токенів) модель показала: чистий JSON — 100.00%, валідний JSON — 100.00%, Accuracy (`final_emotion`) — 76.80%, Accuracy (`final_category`) — 90.20%, Macro F1 — 24.88% та 59.00% відповідно.
- **Метод донавчання й гіперпараметри:** SFT з LoRA (`r=16`, `lora_alpha=32`, `lora_dropout=0.05`, `target_modules="all-linear"`), `learning_rate=2e-4`, `cosine` scheduler, `warmup_steps=10`, `max_steps=150`, ефективний batch size = 16, `assistant_only_loss=True`, `fp16=True`. Тренування тривало 9.63 хв на GPU Tesla T4, пікове використання VRAM склало 4.08 GB, розмір збереженого адаптера — 69.8 MB.
- **Результат після донавчання:** Із коротким промптом (`SHORT_PROMPT`, 37 токенів — скорочення у 4.5 раза) метрики на тому самому тестовому наборі зросли до: чистий JSON — 100.00%, валідний JSON — 100.00%, Accuracy (`final_emotion`) — 91.40% (+14.60%), Accuracy (`final_category`) — 92.20% (+2.00%), Macro F1 — 42.06% (+17.18%) та 75.08% (+16.08%). На загальних генеративних запитах поза задачею проявилося часткове катастрофічне забування (повтори слів та перемикання на польську мову в одному з відповідей), оскільки адаптер тренувався виключно на коротких JSON-відповідях без домішування загальних інструкцій.
- **Аналіз помилок:** Загалом модель помилилася у 60 із 500 тестових прикладів (12.0%). Основні причини помилок:
  1. **Змішаний намір у відгуку:** наприклад, у відгуку *«Гарний вид на центральний Львів, але дуже довго підійматись, треба ліфт»* модель обрала `Suggestion / Idea` замість еталонного `Gratitude / Positive Feedback`, що логічно з огляду на пропозицію в кінці речення.
  2. **Близькі негативні емоції та дисбаланс класів:** модель інколи плутає `Sadness` або `Disgust` із більш частотним класом `Anger`, а нижчий показник Macro F1 для емоцій (42.06%) пояснюється малою кількістю рідкісних емоцій (`Fear`, `Surprise`, `Disgust`) у вибірці.
  3. **Неоднозначність еталонної розмітки:** наприклад, рекомендацію юриста еталон відносить до `Suggestion / Idea`, тоді як модель класифікувала її як `Gratitude / Positive Feedback`.
- **Висновок:** SFT із LoRA дозволив адаптувати компактну модель `Qwen3-1.7B` під прикладну класифікацію українських відгуків на безкоштовному GPU T4 за ~10 хвилин. Точність класифікації та Macro F1 суттєво зросли, формат JSON залишився 100% стабільним, а довжина системного промпту скоротилася зі 168 до 37 токенів. Для використання адаптера у продакшені його варто застосовувати виключно для цільової задачі класифікації (або вимикати адаптер на загальних діалогах).

In [7]:
from huggingface_hub import HfApi, login

login()

repo_id = "darynacherkavska/qwen3-1.7b-ua-reviews-adapter"

api = HfApi()
api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
api.upload_folder(
    folder_path="qwen3-1.7b-ua-reviews-adapter",
    repo_id=repo_id,
    repo_type="model",
)
print(f"Готово! Посилання: https://huggingface.co/{repo_id}")

Готово! Посилання: https://huggingface.co/darynacherkavska/qwen3-1.7b-ua-reviews-adapter


# Опублікований адаптер на Hugging Face Hub

https://huggingface.co/darynacherkavska/qwen3-1.7b-ua-reviews-adapter